In [12]:
import numpy as np
import math
from scipy.constants import m_e, c, e, hbar, physical_constants, epsilon_0, k, pi
from scipy.special import gamma
from scipy import integrate
from tqdm import tqdm
import numba
import matplotlib.pyplot as plt
import pandas as pd
import pdb  # or import ipdb
import csv
from scipy.special import j0
from scipy.special import jv  # For Bessel functions
from scipy.integrate import quad
from scipy.interpolate import interp1d
import h5py

In [13]:
#Multiple species dictionary
ionization_levels = [
    { 'species_name': 'H', 'initial_charge': 0, 'final_charge': 1, 'Uion_eV': 13.6 },
    { 'species_name': 'N', 'initial_charge': 0, 'final_charge': 1, 'Uion_eV': 14.53 },
    { 'species_name': 'N', 'initial_charge': 1, 'final_charge': 2, 'Uion_eV': 29.60 },
    { 'species_name': 'N', 'initial_charge': 2, 'final_charge': 3, 'Uion_eV': 47.45 },
    { 'species_name': 'N', 'initial_charge': 3, 'final_charge': 4, 'Uion_eV': 77.47 },
    { 'species_name': 'N', 'initial_charge': 4, 'final_charge': 5, 'Uion_eV': 97.89 }
]


In [14]:
#calculate all ADK parameters for each transition. The only thing i need is the Z, and the ionization energies and loop through

UH = 13.6 * e
alpha = physical_constants['fine-structure constant'][0]
r_e = physical_constants['classical electron radius'][0]
wa = alpha**3 * c / r_e
Ea = m_e*c**2/e * alpha**4/r_e
for transition_dict in ionization_levels:
    transition_dict['Uion'] = transition_dict['Uion_eV'] * e
    #Z is the charge of the *resulting* ion
    transition_dict['Z'] = transition_dict['final_charge']
    # Calculate effective quantum number n*
    transition_dict['n_eff'] = transition_dict['Z'] * np.sqrt( UH/transition_dict['Uion'] )
    # Assuming l=0, so l_eff = n* - 1
    transition_dict['l_eff'] = transition_dict['n_eff'] - 1
    # C(n*,l*)^2 coefficient
    transition_dict['C2'] = 2**(2*transition_dict['n_eff']) / (transition_dict['n_eff'] * gamma(transition_dict['n_eff']+transition_dict['l_eff']+1) * gamma(transition_dict['n_eff']-transition_dict['l_eff']))
    # Final ADK parameters
    transition_dict['adk_power'] = -(2*transition_dict['n_eff'] - 1)
    transition_dict['adk_prefactor'] = wa * transition_dict['C2'] * ( transition_dict['Uion']/(2*UH) ) \
        * ( 2*(transition_dict['Uion']/UH)**(3./2)*Ea )**(2*transition_dict['n_eff'] - 1)
    transition_dict['adk_exp_prefactor'] = -2./3 * ( transition_dict['Uion']/UH )**(3./2) * Ea

In [22]:

species_keys = ['H_0', 'H_1', 'N_0', 'N_1', 'N_2', 'N_3', 'N_4', 'N_5']
num_species = len(species_keys)
num_transitions = len(ionization_levels)

#arrays to hold the ADK parameters for each transition ( changed to arrays from dictionsries to be compatible with numba.njit )
adk_prefactors = np.array([d['adk_prefactor'] for d in ionization_levels])
adk_powers = np.array([d['adk_power'] for d in ionization_levels])
adk_exp_prefactors = np.array([d['adk_exp_prefactor'] for d in ionization_levels])

# Create arrays that map each transition to its source and target species index
source_indices = np.zeros(num_transitions, dtype=np.int64)
target_indices = np.zeros(num_transitions, dtype=np.int64)
for i, d in enumerate(ionization_levels):
    source_key = f"{d['species_name']}_{d['initial_charge']}"
    target_key = f"{d['species_name']}_{d['final_charge']}"
    source_indices[i] = species_keys.index(source_key)
    target_indices[i] = species_keys.index(target_key)
charges = np.array([0, 1, 0, 1, 2, 3, 4, 5], dtype=np.float64)


@numba.njit
def get_fraction_and_temperature(a0, tau, lambd, ell,
                                 adk_prefactors, adk_powers, adk_exp_prefactors,
                                 source_indices, target_indices, charges,
                                 npts_per_wavelength=80):
    """
    This is the fully-optimized, Numba-jitted version of the function.
    It takes simple NumPy arrays as input for high performance.
    """
    omega = 2*np.pi*c/lambd
    E0 = m_e*omega*c/e
    inv_tau2 = 1./tau**2

    # Initial populations: 95% H, 5% N, all neutral
    populations = np.array([0.95, 0.0, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0])
    kin_energy = 0.0
    t = -3*tau
    dt = lambd/c/npts_per_wavelength

    while (t < 3*tau):
        a_env = a0 * math.exp(-2 * np.log(2) * inv_tau2*t**2)
        a = a_env * math.sqrt( ell[0]**2*np.cos(omega*t)**2 + ell[1]**2*np.sin(omega*t)**2 )
        E = E0 * a_env * math.sqrt( ell[0]**2*np.sin(omega*t)**2 + ell[1]**2*np.cos(omega*t)**2 )

        total_new_electrons = 0.0
        # Loop over every possible ionization transition
        for i in range(len(adk_prefactors)):
            source_idx = source_indices[i]
            target_idx = target_indices[i]
            
            w = 0.0
            if E > 1e-3 and populations[source_idx] > 1e-9:
                w = adk_prefactors[i] * E**adk_powers[i] * math.exp( adk_exp_prefactors[i]/E )

            dp = 1 - math.exp(-w*dt)
            delta_p = dp * populations[source_idx]
            
            populations[source_idx] -= delta_p
            populations[target_idx] += delta_p
            total_new_electrons += delta_p

        if total_new_electrons > 0:
             kin_energy += total_new_electrons * m_e*c**2 * (math.sqrt( 1 + a**2 ) - 1)
        
        t += dt

    T = 0.0
    z_average = np.sum(charges * populations)
    if z_average > 1e-6:
        T = kin_energy / (3/2 * z_average * e)

    return populations, T, t

def calculate_energy_per_length(r_values, Ionfrac, Te, n0=1e24):
    center_idx = len(r_values) // 2
    r_values_centered = r_values[center_idx:] - r_values[center_idx]
    Ionfrac_centered = Ionfrac[center_idx:]
    Te_centered = Te[center_idx:]
    alpha_interp = interp1d(r_values_centered, Ionfrac_centered, kind='linear', bounds_error=False, fill_value=0.0)
    Te_interp = interp1d(r_values_centered, Te_centered, kind='linear', bounds_error=False, fill_value=0.0)

    def integrand(r):
        k = 1.380649e-23
        alpha = alpha_interp(r)
        Te_ev = Te_interp(r)
        Te_K = Te_ev * 11604
        result = n0 * alpha * 1.5 * k * Te_K * 2 * np.pi * r
        return result

    r_max = 0.0396
    total_integral, _ = quad(integrand, 0, r_max, limit=200)
    print(f"\nIntegration results:")
    print(f"Integral value (J/m): {total_integral:.2e}")
    energy_per_length_erg_cm = total_integral * 1e7 / 100
    print(f"Final energy (erg/cm): {energy_per_length_erg_cm:.2e}")
    return energy_per_length_erg_cm

def process_intensity_profile(h5_file_path, output_file='plasma_data_multi_species.csv'):
    with h5py.File(h5_file_path, 'r') as f:
        # Fix for potential h5py precision error
        intensity_data = np.zeros(f['intensity_profiles'].shape[1:], dtype=np.float64)
        f['intensity_profiles'].read_direct(intensity_data, source_sel=np.s_[0,:])
        intensity_profile = intensity_data / 1.67 * 0.38
        
        lambd = 0.8e-6
        tau = 40e-15
        a0 = e * lambd / (pi * m_e * c) * np.sqrt(intensity_profile / (2 * epsilon_0 * c**3))
        print("Max Intensity (W/cm^2): ", np.max(intensity_profile)/1e4)

        Te = np.zeros_like(a0)
        All_Populations = np.zeros((len(a0), num_species))
        ell = [1.0, 0.0]

        # Loop over the radial profile of the laser
        for i in tqdm(range(len(a0))):
            # Call the MODIFIED,JITTED function passing the new numpy arrays
            final_populations, T, _ = get_fraction_and_temperature(a0[i], tau, lambd, ell,
                                                                   adk_prefactors, adk_powers, adk_exp_prefactors,
                                                                   source_indices, target_indices, charges)
            Te[i] = T
            All_Populations[i,:] = final_populations

        # --- Post-processing ---
        r_values = np.linspace(-3.96e-2, 3.96e-2, len(a0))
        # Calculate the average ionization <Z> profile for plotting and energy calculation
        z_average_profile = np.sum(All_Populations * charges, axis=1)


        # =================================================================
        # PRINT ON-AXIS NITROGEN POPULATIONS
        # =================================================================
        print("\n--- On-Axis (Peak Intensity) Final Populations ---")
        center_idx = len(a0) // 2
        on_axis_populations = All_Populations[center_idx, :]

        # Loop through and print only the nitrogen species (indices 2 through 7)
        for i in range(2, len(species_keys)):
            species_name = species_keys[i].replace('_0',' (neu)').replace('_','+')
            population = on_axis_populations[i]
            # Only print if the population is significant to avoid clutter
            if population > 1e-6:
                print(f"{species_name:<12}: {population:.6f}")
        print("--------------------------------------------------\n")
        # =================================================================

        
        #THis is suspect, I'm not sure the energy per length I get is accurate
        energy_per_length = calculate_energy_per_length(r_values, z_average_profile, Te)

        if output_file:
            mid_point = len(a0) // 2
            # Create a dictionary for the DataFrame, starting with radius and temp
            data_dict = {
                'Radius (cm)': (r_values * 100)[mid_point:],
                'Electron Temperature (K)': (Te * 11604)[mid_point:]
            }
            # Add a column for each species' population
            for i, key in enumerate(species_keys):
                data_dict[key] = All_Populations[mid_point:, i]

            df = pd.DataFrame(data_dict)
            df.to_csv(output_file, index=False, float_format='%.4e')
            print(f"Detailed species data saved to {output_file}")

# --- Plotting ---
        
        fig, (ax_h, ax_n) = plt.subplots(
            nrows=2, ncols=1, figsize=(8, 10), sharex=True
        )
        fig.suptitle('Multi-Species Ionization and Heating\nZ = 22cm', fontsize=16)

        center_idx = len(r_values) // 2
        r_plot = r_values[center_idx:] * 1e6 # radius in microns

        # --- Plot 1: Hydrogen Species ---
        ax_h.set_title("Hydrogen Species")
        h0_idx = species_keys.index('H_0')
        h1_idx = species_keys.index('H_1')
        ax_h.plot(r_plot, All_Populations[center_idx:, h0_idx], label='H (neutral)')
        ax_h.plot(r_plot, All_Populations[center_idx:, h1_idx], label='H+')
        ax_h.set_ylabel('Population Fraction')
        ax_h.set_ylim(-0.05, 1.05)
        ax_h.grid(True, linestyle=':')
        ax_h.legend(loc='center right')

        # Add temperature to the hydrogen plot on a secondary y-axis
        ax_h_temp = ax_h.twinx()
        ax_h_temp.plot(r_plot, Te[center_idx:], color='red', alpha=0.6)
        ax_h_temp.set_ylabel('Te (eV)', color='red')
        ax_h_temp.tick_params(axis='y', labelcolor='red')
        ax_h_temp.set_ylim(bottom=0)

# --- Plot 2: Nitrogen Species (Stacked Area Plot) ---
        ax_n.set_title("Nitrogen Species Composition")

        # Prepare the data for the stacked plot: a list of 1D arrays for each N species
        n_species_indices = range(2, len(species_keys)) # Indices for N_0, N_1, ...
        y_stack = [All_Populations[center_idx:, i] for i in n_species_indices]
        stack_labels = [species_keys[i].replace('_0',' (neu)').replace('_','+') for i in n_species_indices]

        # Use a colormap for visually distinct colors ( used gemini for this )
        colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(y_stack)))

        # Create the stacked plot
        ax_n.stackplot(r_plot, y_stack, labels=stack_labels, colors=colors, alpha=0.8)

        ax_n.set_xlabel('Position (μm)')
        ax_n.set_ylabel('Population Fraction')
        # The y-limit should be slightly larger than the total dopant percentage
        ax_n.set_ylim(bottom=0, top=0.011)
        ax_n.grid(True, linestyle=':')
        # Place legend to the right of the plot to avoid covering data
        ax_n.legend(loc='center left', bbox_to_anchor=(1.18, 0.5))

        # Add laser intensity to the nitrogen plot on a secondary y-axis
        ax_n_int = ax_n.twinx()
        # The intensity is in W/m^2. Convert to 10^14 W/cm^2 for plotting.
        intensity_plot = intensity_profile[center_idx:] * 1e-18
        ax_n_int.plot(r_plot, intensity_plot, color='gray', linestyle='--', alpha=0.8)
        ax_n_int.set_ylabel('Intensity (10¹⁴ W/cm²)', color='gray')
        ax_n_int.tick_params(axis='y', labelcolor='gray')
        ax_n_int.set_ylim(bottom=0)
        
        # Adjust the main plot's right margin to make space for the legend
        fig.subplots_adjust(right=0.75)

        # Set shared x-axis limits
        plt.xlim(0, 400)
        # Adjust layout to make room for the main title
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
        return Te, All_Populations, a0, energy_per_length

if __name__ == "__main__":
    h5_file_path = "combined_simulation.h5"
    Te, All_Pops, a0, epsilon = process_intensity_profile(h5_file_path)


Max Intensity (W/cm^2):  5165990002241257.0


  2%|█                                                   | 2723/131999 [00:02<01:38, 1306.23it/s]


SystemError: CPUDispatcher(<function get_fraction_and_temperature at 0x2d8a49940>) returned a result with an exception set